# Assessment 2: Pemodelan Machine Learning (Klasifikasi & Klastering) — Revisi

**Dataset:** New York City Airbnb 2019

Notebook ini berisi implementasi lengkap dari pipa pemrosesan data (preprocessing pipeline), pemodelan Supervised Learning (Klasifikasi Popularitas), dan Unsupervised Learning (Klastering Geografis & Ekonomi) dengan validasi silang, penyetelan ambang keputusan, dan pemilihan konfigurasi klastering secara objektif.


In [ ]:
%pip install scikit-learn
import warnings
warnings.filterwarnings("ignore")

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


from IPython.display import Markdown, display

RANDOM_STATE = 42

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)

def section(title):
    display(Markdown(f"## {title}"))

def ask_plot(question):
    display(Markdown(f"**Pertanyaan grafik:** {question}"))

def explain_plot(points):
    display(Markdown(f"**Interpretasi:**\n" + "\n".join(f"- {p}" for p in points)))

def fmt_num(x, digits=2):
    if pd.isna(x):
        return "NA"
    if abs(x) >= 1000:
        return f"{x:,.{digits}f}"
    return f"{x:.{digits}f}"


In [2]:
# Unduh / muat dataset dengan fallback ke file lokal
selected_cols = [
    "id", "host_id",
    "neighbourhood_group", "neighbourhood",
    "latitude", "longitude",
    "room_type",
    "price", "minimum_nights",
    "number_of_reviews", "last_review", "reviews_per_month",
    "calculated_host_listings_count", "availability_365",
]

try:
    path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
    csv_path = f"{path}/AB_NYC_2019.csv"
    print(f"[kagglehub] Dataset diunduh: {path}")
except Exception:
    csv_path = "AB_NYC_2019.csv"
    print(f"[local] Menggunakan file lokal: {csv_path}")

df_raw = pd.read_csv(csv_path, usecols=selected_cols, parse_dates=["last_review"])

# Filter data tidak valid
mask_valid = (
    (df_raw["price"] > 0)
    & (df_raw["availability_365"].between(0, 365))
    & (df_raw["latitude"].between(40.45, 40.95))
    & (df_raw["longitude"].between(-74.30, -73.65))
)
df = df_raw.loc[mask_valid].copy()

print(f"Baris setelah pembersihan: {len(df):,}")
print(f"Baris terbuang: {len(df_raw) - len(df):,}")


### 0. Feature Engineering untuk Clustering

**Tujuan:** Menyiapkan fitur numerik siap-klastering dari data mentah Airbnb NYC 2019.

Transformasi yang dilakukan:
1. **Log-transform** — menormalkan distribusi `price`, `minimum_nights`, `number_of_reviews` yang sangat skewed (power-law).
2. **Haversine distance** — menghitung jarak tiap listing ke pusat NYC (Times Square) dalam km.
3. **Capping & winsorize** — memotong nilai ekstrem di `availability_365` dan `minimum_nights` agar outlier tidak mendominasi.
4. **Recency & binary flags** — umur review terakhir dan indikator apakah listing pernah direview.
5. **Drop redundant** — menghapus `reviews_per_month` karena berkorelasi tinggi dengan `number_of_reviews`.

Hasil feature engineering akan dipakai oleh **keempat kandidat** klastering secara konsisten.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# FEATURE ENGINEERING — NYC Airbnb 2019
# ———————————————————————————————————————————————————————————————————————

import numpy as np               # Operasi numerik untuk transformasi fitur

# ----- 1. Tanggal referensi -----
REF_DATE = df['last_review'].max()                         # Review paling baru di dataset sebagai acuan
print(f"Tanggal referensi (review terbaru): {REF_DATE.date()}")  # Verifikasi

# ----- 2. Log-transform harga -----
df['log_price'] = np.log(df['price'])             # log(price): menormalkan distribusi harga skewed

# ----- 3. Log-transform minimum_nights -----
df['log_min_nights'] = np.log1p(df['minimum_nights'])  # log1p(x) = log(1+x), aman untuk nilai 0

# ----- 4. Log-transform jumlah review -----
df['log_number_of_reviews'] = np.log1p(df['number_of_reviews'])  # log1p untuk listing tanpa review

# ----- 5. Umur review terakhir (hari) -----
df['last_review_age_days'] = (REF_DATE - df['last_review']).dt.days  # Selisih hari dari REF_DATE
df['last_review_age_days'] = df['last_review_age_days'].fillna(9999)  # NaN → 9999: belum pernah direview

# ----- 6. Flag binary: listing punya review -----
df['has_reviews'] = (df['number_of_reviews'] > 0).astype(int)  # 1 = pernah, 0 = belum direview

# ----- 7. Jarak ke pusat NYC (Times Square) -----
NYC_CENTER = (40.7580, -73.9855)                             # Koordinat Times Square

def haversine_km(lat1, lon1, lat2, lon2):
    """Hitung jarak great-circle antara dua titik (km) — Haversine formula."""
    R = 6371.0                                                # Radius bumi (km)
    dlat = np.radians(lat2 - lat1)                            # Δ latitude → radian
    dlon = np.radians(lon2 - lon1)                            # Δ longitude → radian
    a = (np.sin(dlat/2)**2                                     # Haversine term-a
         + np.cos(np.radians(lat1))
         * np.cos(np.radians(lat2))
         * np.sin(dlon/2)**2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))           # Haversine term-c
    return R * c                                              # Jarak akhir (km)

df['distance_to_center_km'] = haversine_km(
    df['latitude'], df['longitude'],   # Koordinat tiap listing
    NYC_CENTER[0], NYC_CENTER[1]        # Koordinat pusat NYC
)

# ----- 8. Capping availability_365 -----
df['availability_365_capped'] = df['availability_365'].clip(0, 365)  # Pastikan dalam [0, 365]

# ----- 9. Winsorize minimum_nights (Q99) -----
Q99 = df['minimum_nights'].quantile(0.99)           # Hitung percentile ke-99
df['minimum_nights_capped'] = df['minimum_nights'].clip(0, Q99)  # Potong di Q99

# ----- 10. Drop fitur redundant -----
df = df.drop(columns=['reviews_per_month'])          # Korelasi tinggi dengan number_of_reviews

# ----- Verifikasi -----
print(f"Kolom setelah FE : {len(df.columns)}")       # Jumlah kolom
print(f"Baris           : {len(df):,}")               # Jumlah baris
print(f"Missing values  : {df.isnull().sum().sum()}") # Total NaN (harus 0)
num_cols = df.select_dtypes(include='number')
print(f"Infinite values : {np.isinf(num_cols).sum().sum()}")  # Total inf (harus 0)
# Tampilkan statistik fitur kunci hasil engineering
df[['price','log_price','minimum_nights','log_min_nights',
    'number_of_reviews','log_number_of_reviews','distance_to_center_km']].describe()


In [5]:
df.head()

## UNSUPERVISED LEARNING — Multi-Perspective Clustering

Notebook ini mengimplementasikan **empat perspektif klastering** pada dataset NYC Airbnb 2019
(48.884 listing setelah cleansing).  Masing-masing perspektif menangkap dimensi yang berbeda:

| Kandidat | Nama | Dimensi Utama | Algoritma |
|---|---|---|---|
| **A** | Geographic Zone Discovery | Spasial (lat, lon) | KMeans + DBSCAN |
| **B** | Price Tier Segmentation | Ekonomi (harga, demand) | KMeans |
| **C** | Host Archetypes | Perilaku Host | KMeans + Agglomerative |
| **D** | Multi-Dimensional Market Segmentation | Semua dimensi | KMeans + GMM |

**Metrik validasi:** **Silhouette Score** dipilih sebagai metrik tunggal (best practice).
Range pengujian k = 2–10.
Ambang minimum interpretable: Silhouette ≥ 0.25.
Di bawah angka ini, struktur klaster dianggap lemah dan dilaporkan apa adanya.

*Random state seluruh eksperimen:* `RANDOM_STATE = 42`.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# HELPER FUNCTIONS — reusable untuk keempat kandidat
# ———————————————————————————————————————————————————————————————————————

from sklearn.preprocessing import RobustScaler, StandardScaler      # Scaling yang tahan outlier
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering  # Algoritma klastering
from sklearn.mixture import GaussianMixture                          # GMM (soft clustering)
from sklearn.metrics import silhouette_score                         # Metrik validasi utama
from sklearn.neighbors import NearestNeighbors                        # k-distance untuk DBSCAN
from sklearn.decomposition import PCA                                 # Reduksi dimensi → visualisasi
from scipy.cluster.hierarchy import linkage, dendrogram               # Dendrogram untuk AHC
import matplotlib.pyplot as plt                                       # Plotting
import seaborn as sns                                                 # Styling
import pandas as pd                                                   # DataFrame
import numpy as np                                                    # Numerik

# ——— Fungsi 1: Evaluasi Silhouette Score ———————————————————————————
def evaluate_k_silhouette(X, k_range, random_state=RANDOM_STATE, sample_size=10000):
    """Hitung Silhouette Score untuk KMeans pada berbagai k.
    sample_size=10000: batas jumlah sampel utk perhitungan (hemat RAM, default sklearn=None).
    Return: (list_scores, best_k, k_range)"""
    scores = []                                            # Inisialisasi list skor
    for k in k_range:                                      # Iterasi kandidat k
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)  # Inisialisasi KMeans
        labels = km.fit_predict(X)                         # Fit + prediksi label
        n = min(sample_size, len(X))                         # Batasi sampel utk hemat memori
        sil = silhouette_score(X, labels, sample_size=n)   # Hitung Silhouette Score (sample)
        scores.append(sil)                                 # Simpan skor
    best_k = k_range[int(np.argmax(scores))]               # Ambil k dengan skor tertinggi
    return scores, best_k, k_range                         # Return tuple hasil


# ——— Fungsi 2: Plot Silhouette vs k ———————————————————————————————
def plot_silhouette_scan(k_range, scores, best_k, title='Silhouette Score vs k'):
    """Plot kurva Silhouette Score sebagai fungsi jumlah klaster."""
    fig, ax = plt.subplots(figsize=(10, 5))                # Buat figure
    ax.plot(k_range, scores, marker='o', linewidth=2, color='steelblue')  # Garis + titik
    ax.axvline(x=best_k, color='crimson', linestyle='--', alpha=0.7)    # Garis di k optimal
    ax.scatter(best_k, max(scores), color='crimson', s=120, zorder=5)   # Tandai titik optimal
    ax.set_xlabel('Jumlah Klaster (k)')                    # Label sumbu-x
    ax.set_ylabel('Silhouette Score')                      # Label sumbu-y
    ax.set_title(title)                                    # Judul
    ax.set_xticks(k_range)                                 # Ticks integer di tiap k
    ax.grid(True, alpha=0.3)                               # Grid tipis
    plt.tight_layout()                                     # Compact layout
    plt.show()                                             # Tampilkan
    print(f"   k optimal = {best_k}  |  Silhouette = {max(scores):.4f}")


# ——— Fungsi 3: Plot peta NYC berwarna klaster —————————————————————
def plot_map_clusters(df, labels, title='Peta NYC — Klaster', cmap='tab10',
                       figsize=(12, 10), s=1, alpha=0.6):
    """Scatter plot peta NYC: longitude vs latitude, warna = label klaster."""
    fig, ax = plt.subplots(figsize=figsize)                # Buat figure peta
    scatter = ax.scatter(
        df['longitude'], df['latitude'],                   # Koordinat listing
        c=labels, cmap=cmap, s=s, alpha=alpha              # Warna, ukuran, transparansi
    )
    ax.set_xlabel('Longitude')                             # Label sumbu-x
    ax.set_ylabel('Latitude')                              # Label sumbu-y
    ax.set_title(title)                                    # Judul
    # Legend warna per klaster
    legend1 = ax.legend(*scatter.legend_elements(), title='Klaster', loc='upper right')
    ax.add_artist(legend1)                                 # Tampilkan legend
    # Koreksi aspect ratio berdasarkan latitude (meridian convergence)
    lat_mean = df['latitude'].mean()                       # Rata-rata latitude
    ax.set_aspect(1 / np.cos(np.radians(lat_mean)))        # Aspect seimbang geografis
    plt.tight_layout()                                     # Compact layout
    plt.show()                                             # Tampilkan


# ——— Fungsi 4: Tabel profil klaster ———————————————————————————————
def profile_clusters(df_orig, labels, feature_cols):
    """Hitung median + count per klaster → tabel profil."""
    p = df_orig.copy()                                     # Copy dataframe
    p['cluster'] = labels                                  # Tambah kolom label
    prof = p.groupby('cluster')[feature_cols].median()     # Median per fitur per klaster
    prof['count'] = p.groupby('cluster').size()             # Jumlah listing per klaster
    prof['pct(%)'] = (prof['count'] / len(p) * 100).round(1)  # Persentase
    return prof                                            # Return tabel profil


# ——— Fungsi 5: Plot 2 panel peta (KMeans vs DBSCAN) ——————————————
def plot_map_compare(df, labels1, labels2, title1='KMeans', title2='DBSCAN',
                       cmap='tab10', figsize=(18, 8)):
    """Dua peta NYC berdampingan untuk membandingkan dua hasil klastering."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize) # Buat 1 baris 2 kolom
    # Panel kiri
    sc1 = ax1.scatter(df['longitude'], df['latitude'],     # Scatter KMeans
                      c=labels1, cmap=cmap, s=1, alpha=0.6)
    ax1.set_title(title1)                                  # Judul panel 1
    ax1.set_xlabel('Longitude')                            # Label x
    ax1.set_ylabel('Latitude')                             # Label y
    legend1 = ax1.legend(*sc1.legend_elements(), title='Klaster', loc='upper right')
    ax1.add_artist(legend1)
    # Panel kanan
    sc2 = ax2.scatter(df['longitude'], df['latitude'],     # Scatter DBSCAN
                      c=labels2, cmap=cmap, s=1, alpha=0.6)
    ax2.set_title(title2)                                  # Judul panel 2
    ax2.set_xlabel('Longitude')                            # Label x
    # Legend khusus DBSCAN (include noise = -1)
    handles, _legend_labels = sc2.legend_elements()
    ax2.legend(handles, [f'Klaster {i}' if int(i) >= 0 else 'Noise' for i, _ in enumerate(handles)],
               title='Klaster', loc='upper right')
    # Aspect ratio geografis untuk kedua panel
    lat_mean = df['latitude'].mean()
    ax1.set_aspect(1 / np.cos(np.radians(lat_mean)))
    ax2.set_aspect(1 / np.cos(np.radians(lat_mean)))
    plt.tight_layout()
    plt.show()

print("Semua helper functions siap.")  # Verifikasi


### KANDIDAT A — Geographic Zone Discovery

**Perspektif Akademis:**

Klastering geografis merupakan analisis **spatial point-pattern clustering** dalam
bidang urban analytics.  Tujuan: menemukan zona-zona alami (natural zones) yang
mungkin **berbeda** dari batasan administratif 5 borough NYC.

**Landasan teori:**
- *Chainey & Ratcliffe (2005)* — Hot-spot analysis dan zonasi berbasis densitas
  dapat mengungkap pola yang tidak terlihat di peta administratif.
- *Rey & Anselin (2009)* — Spatial autocorrelation (ESDA) memperlihatkan bahwa
  fenomena urban sering membentuk klaster yang tidak selaras dengan batas ward/borough.

**Pertanyaan penelitian:**
Apakah listings Airbnb di NYC membentuk zona alami yang berbeda dari 5 borough?
Apakah DBSCAN bisa mendeteksi "listing suburban" yang sparse sebagai noise?

**Fitur:** `latitude`, `longitude` setelah `RobustScaler`.

**Algoritma:**
- **KMeans** — zona spherical (sama-radius dari centroid).
- **DBSCAN** — zona berbasis kepadatan; otomatis mendeteksi listing terisolasi sebagai *noise*.

**Metrik evaluasi:** Silhouette Score (KMeans).  DBSCAN dievaluasi secara kualitatif
(kemampuan deteksi noise dan kesesuaian visual di peta).


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT A — Persiapan Data Geografis
# ———————————————————————————————————————————————————————————————————————

# Pilih fitur untuk Kandidat A
features_A = ['latitude', 'longitude']                      # Hanya koordinat geografis

# Ekstrak matriks fitur
X_geo_raw = df[features_A].values                           # Konversi ke numpy array (N, 2)

# Scaling dengan RobustScaler (tahan outlier koordinat)
scaler_A = RobustScaler()                                   # Inisialisasi scaler
X_geo = scaler_A.fit_transform(X_geo_raw)                   # Fit + transform: median-centered, IQR-scaled

print(f"Matriks fitur A  : {X_geo.shape}")                  # Dimensi: (48884, 2)
print(f"Range setelah scaling: [{X_geo.min():.2f}, {X_geo.max():.2f}]")  # Cek rentang


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT A — Silhouette Scan (pilih k optimal)
# ———————————————————————————————————————————————————————————————————————

k_range = range(2, 11)                                       # Uji k = 2 s.d. 10

# Panggil fungsi evaluasi Silhouette
scores_A, best_k_A, _ = evaluate_k_silhouette(
    X_geo, k_range, random_state=RANDOM_STATE               # Matriks, rentang k, seed
)

# Plot hasil scan
plot_silhouette_scan(
    k_range=k_range,                                         # Rentang k
    scores=scores_A,                                         # List skor Silhouette
    best_k=best_k_A,                                         # k optimal
    title='Kandidat A: Silhouette Score — Geographic Zone Discovery'
)


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT A — Model Fit: KMeans + DBSCAN
# ———————————————————————————————————————————————————————————————————————

# ---- KMeans dengan k optimal ----
kmeans_A = KMeans(n_clusters=best_k_A, random_state=RANDOM_STATE, n_init=10)  # Inisialisasi
labels_kmeans_A = kmeans_A.fit_predict(X_geo)                                  # Fit + prediksi

# ---- DBSCAN: cari eps via k-distance elbow ----
k_neighbors = 5                                                    # Jumlah tetangga untuk k-distance
neigh = NearestNeighbors(n_neighbors=k_neighbors)                   # Inisialisasi NearestNeighbors
neigh.fit(X_geo)                                                    # Fit pada data geografis
distances_all, _ = neigh.kneighbors(X_geo)                          # Ambil jarak ke k-tetangga
k_dist = np.sort(distances_all[:, k_neighbors - 1])                 # Jarak ke tetangga ke-k, diurutkan

# Deteksi elbow dari k-distance menggunakan maximum curvature
deriv1 = np.diff(k_dist)                                            # Turunan pertama
deriv2 = np.diff(deriv1)                                            # Turunan kedua (curvature)
elbow_idx = int(np.argmax(deriv2)) + 1                               # Indeks max curvature
eps_val = k_dist[elbow_idx]                                         # Nilai eps di elbow

print(f"DBSCAN eps (auto elbow) : {eps_val:.4f}")                   # Cetak eps yang ditemukan

# ---- DBSCAN fit ----
min_samples_DB = 50                                                 # Minimal tetangga (cukup besar untuk data ini)
dbscan_A = DBSCAN(eps=eps_val, min_samples=min_samples_DB)           # Inisialisasi DBSCAN
labels_dbscan_A = dbscan_A.fit_predict(X_geo)                        # Fit + prediksi

# Hitung statistik DBSCAN
n_clusters_db = len(set(labels_dbscan_A)) - (1 if -1 in labels_dbscan_A else 0)  # Jumlah klaster (excl noise)
n_noise = int(np.sum(labels_dbscan_A == -1))                                       # Jumlah noise
print(f"DBSCAN klaster : {n_clusters_db}")                           # Cetak jumlah klaster
print(f"DBSCAN noise   : {n_noise} ({n_noise/len(labels_dbscan_A)*100:.1f}%)")  # Cetak jumlah & % noise

# ---- Silhouette untuk DBSCAN (jika ≥2 klaster valid) ----
valid_mask = labels_dbscan_A != -1                                   # Filter noise
if n_clusters_db >= 2:
    n_dbscan = int(valid_mask.sum())                                 # Jumlah titik non-noise
    sil_dbscan = silhouette_score(
        X_geo[valid_mask],                                           # Data non-noise
        labels_dbscan_A[valid_mask],                                 # Label non-noise
        sample_size=min(10000, n_dbscan)                             # Hemat memori utk data besar
    )
    print(f"DBSCAN Silhouette (excl noise): {sil_dbscan:.4f}")
else:
    sil_dbscan = None
    print("DBSCAN: hanya 1 klaster — Silhouette tidak dapat dihitung.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT A — Peta NYC: KMeans vs DBSCAN
# ———————————————————————————————————————————————————————————————————————

plot_map_compare(
    df,                                                             # DataFrame asli (koordinat)
    labels1=labels_kmeans_A,                                         # Label KMeans
    labels2=labels_dbscan_A,                                         # Label DBSCAN
    title1=f'KMeans — k={best_k_A}',                                 # Judul panel kiri
    title2=f'DBSCAN (eps={eps_val:.3f}, min_samples={min_samples_DB})',  # Judul panel kanan
    cmap='tab10'                                                     # Colormap 10 warna
)
print("Interpretasi peta:")
print("  • Panel kiri (KMeans): zona spherical — centroid di pusat zona.")
print("  • Panel kanan (DBSCAN): zona berbasis kepadatan — titik abu-abu = noise (listing terisolasi).")
print("  • DBSCAN mendeteksi listing di pinggiran/suburban yang tidak membentuk klaster padat.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT A — Tabel Profil Klaster
# ———————————————————————————————————————————————————————————————————————

# Fitur untuk profil Kandidat A
profile_features_A = ['price', 'distance_to_center_km', 'number_of_reviews',
                       'availability_365_capped', 'calculated_host_listings_count']

# Buat tabel profil untuk KMeans
profile_A = profile_clusters(df, labels_kmeans_A, profile_features_A)

# Tampilkan tabel
display(profile_A.style
    .format("{:.1f}", subset=profile_features_A)            # Format 1 desimal
    .background_gradient(cmap='Blues', subset=profile_features_A)  # Gradasi biru
    .set_caption(f'Kandidat A — Profil KMeans (k={best_k_A})')  # Judul tabel
)

print(f"Silhouette (KMeans) : {scores_A[best_k_A - k_range[0]]:.4f}")  # Skor dari scan sebelumnya


### KANDIDAT B — Price Tier Segmentation

**Perspektif Akademis:**

Segmentasi harga adalah aplikasi klasik dari **market segmentation theory** (Smith, 1956)
dalam konteks hospitality.  Setiap listing Airbnb memiliki *value proposition* yang
tercermin dalam harga, ketersediaan, permintaan, dan lokasi.

**Landasan teori:**
- *Dolnicar (2002)* — Market segmentation di sektor pariwisata menunjukkan bahwa
  wisatawan memilih akomodasi berdasarkan *value bundles*, bukan hanya harga.
- *RFM (Recency-Frequency-Monetary)* — Kerangka dari e-commerce: recency
  (`last_review_age_days`), frequency (`reviews_per_month` → diganti `log_number_of_reviews`),
  monetary (`log_price`).

**Pertanyaan penelitian:**
Berapa tier harga natural yang terbentuk di NYC? Apakah tier-tersebut berkorelasi
dengan jarak dari pusat kota?

**Fitur:**
- `log_price` — besaran moneter (value)
- `availability_365_capped` — ketersediaan (supply)
- `log_number_of_reviews` — intensitas demand
- `last_review_age_days` — recency demand
- `distance_to_center_km` — lokasi (bukan koordinat, agar dimensi geografis direduksi jadi 1)

**Algoritma:** KMeans — cocok untuk segmentasi diskret dengan fitur kontinu.

**Metrik evaluasi:** Silhouette Score.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT B — Persiapan Data Ekonomi
# ———————————————————————————————————————————————————————————————————————

# Pilih fitur ekonomi + lokasi (tanpa lat/lon mentah)
features_B = [
    'log_price',                    # Log harga: proxy nilai
    'availability_365_capped',      # Ketersediaan setahun
    'log_number_of_reviews',        # Log demand intensity
    'last_review_age_days',         # Recency demand (hari sejak review terakhir)
    'distance_to_center_km',        # Jarak ke Times Square
]

# Ekstrak + scale
X_econ_raw = df[features_B].values                              # Konversi ke numpy
scaler_B = RobustScaler()                                       # Scaler tahan outlier
X_econ = scaler_B.fit_transform(X_econ_raw)                      # Fit + transform

print(f"Matriks fitur B : {X_econ.shape}")                       # Dimensi
print(f"Fitur: {features_B}")                                    # Daftar fitur


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT B — Silhouette Scan
# ———————————————————————————————————————————————————————————————————————

scores_B, best_k_B, _ = evaluate_k_silhouette(
    X_econ, k_range, random_state=RANDOM_STATE
)

plot_silhouette_scan(
    k_range=k_range,
    scores=scores_B,
    best_k=best_k_B,
    title='Kandidat B: Silhouette Score — Price Tier Segmentation'
)


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT B — Model Fit KMeans
# ———————————————————————————————————————————————————————————————————————

kmeans_B = KMeans(n_clusters=best_k_B, random_state=RANDOM_STATE, n_init=10)
labels_B = kmeans_B.fit_predict(X_econ)

sil_B = silhouette_score(X_econ, labels_B, sample_size=10000)    # Konfirmasi skor akhir (sample utk hemat RAM)
print(f"KMeans k={best_k_B} — Silhouette = {sil_B:.4f}")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT B — Visualisasi: Peta + Boxplot
# ———————————————————————————————————————————————————————————————————————

fig, (ax_map, ax_box) = plt.subplots(1, 2, figsize=(20, 8))     # 1 baris, 2 panel

# --- Panel 1: Peta NYC dengan warna tier harga ---
scatter_B = ax_map.scatter(
    df['longitude'], df['latitude'],                             # Koordinat
    c=labels_B, cmap='tab10', s=0.8, alpha=0.5                  # Warna per tier
)
ax_map.set_title(f'Kandidat B — Price Tier (k={best_k_B})')     # Judul
ax_map.set_xlabel('Longitude')
ax_map.set_ylabel('Latitude')
legend_B = ax_map.legend(*scatter_B.legend_elements(), title='Tier', loc='upper right')
ax_map.add_artist(legend_B)
lat_mean = df['latitude'].mean()
ax_map.set_aspect(1 / np.cos(np.radians(lat_mean)))

# --- Panel 2: Boxplot harga per tier ---
df_tmp = df.copy()                                                # Copy sementara
df_tmp['tier'] = labels_B.astype(str)                             # Label sebagai string
ordered = sorted(df_tmp['tier'].unique(), key=lambda x: int(x))   # Urutkan tier
sns.boxplot(data=df_tmp, x='tier', y='price', order=ordered,      # Boxplot
            palette='tab10', showfliers=False, ax=ax_box)         # Sembunyikan outlier
ax_box.set_title('Distribusi Harga per Tier')                     # Judul
ax_box.set_xlabel('Price Tier')                                   # Label x
ax_box.set_ylabel('Price (USD)')                                  # Label y
ax_box.set_yscale('log')                                          # Skala log

plt.tight_layout()
plt.show()
print("Panel kiri: peta sebaran tier. Panel kanan: distribusi harga per tier (log-scale).")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT B — Tabel Profil Tier Harga
# ———————————————————————————————————————————————————————————————————————

profile_features_B = ['price', 'log_price', 'distance_to_center_km',
                       'number_of_reviews', 'availability_365_capped',
                       'minimum_nights_capped']

profile_B = profile_clusters(df, labels_B, profile_features_B)

# Urutkan berdasarkan median price untuk interpretasi tier
profile_B = profile_B.sort_values('price')                       # Urut naik: Budget → Luxury

# Beri nama tier berdasarkan urutan
tier_names = {i: f'T{idx+1}' for idx, i in enumerate(profile_B.index)}
profile_B.index = [f'{tier_names[i]} ({["Budget","Value","Mid","Premium","Luxury"][idx] if idx < 5 else f"T{idx+1}"})'
                    for idx, i in enumerate(profile_B.index)]

display(profile_B.style
    .format("{:.1f}", subset=profile_features_B)
    .background_gradient(cmap='YlOrRd', subset=profile_features_B)
    .set_caption(f'Kandidat B — Profil Price Tier (k={best_k_B})')
)

# Interpretasi
print(f"Silhouette Price Tier: {sil_B:.4f}")
print(f"Jumlah tier: {best_k_B}")
print("Semakin kecil Silhouette (< 0.25) → overlap antar tier tinggi (harga kontinu).")


### KANDIDAT C — Host Archetypes

**Perspektif Akademis:**

Identifikasi arketipe host adalah analisis **supply-side segmentation** pada
marketplace two-sided.  Airbnb memiliki dua jenis host fundamental:

- **Peer-to-peer (occasional)** — menyewakan 1–2 listing sebagai pendapatan sampingan.
- **Professional (multi-listing)** — mengelola banyak listing sebagai bisnis utama.

**Landasan teori:**
- *Zervas, Proserpio & Byers (2017)* — "The Rise of the Sharing Economy" mendokumentasikan
  transisi dari murni peer-to-peer ke host profesional yang mengubah dinamika pasar.
- *Airbnb Internal Research (2019)* — ~65% host di NYC adalah single-listing, tapi
  multi-listing host mendominasi supply dan revenue.
- *Kooti et al. (2017)* — Host profesional berperilaku berbeda: minimum nights lebih rigid,
  response rate lebih tinggi, ketersediaan lebih terencana.

**Pertanyaan penelitian:**
Apa saja arketipe host berdasarkan perilaku listing? Apakah ada struktur hierarki
(misal: Host Profesional → Profesional Massal vs Boutique)?

**Fitur:**
- `calculated_host_listings_count` — proksi profesionalisme (1 = individual)
- `log_min_nights` — komitmen rental (long-stay vs short-stay)
- `log_number_of_reviews` — aktivitas/kredibilitas listing
- `last_review_age_days` — recency (listing aktif vs dorman)

**Algoritma:**
- **KMeans** — arketipe diskret.
- **Agglomerative Clustering (Ward)** — analisis hierarki: apakah arketipe punya sub-arketipe?

**Metrik evaluasi:** Silhouette Score. Dendrogram untuk validasi visual struktur hierarki.

**Catatan peta:** Dimensi host **bukan dimensi geografis**.  Peta hanya menunjukkan
*di mana* arketipe tertentu dominan; interpretasi utama ada pada tabel profil.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT C — Persiapan Data Host
# ———————————————————————————————————————————————————————————————————————

features_C = [
    'calculated_host_listings_count',  # Jumlah listing oleh host (1 = individu)
    'log_min_nights',                  # Log min nights (komitmen rental)
    'log_number_of_reviews',           # Log jumlah review (aktivitas)
    'last_review_age_days',            # Umur review terakhir (recency)
]

X_host_raw = df[features_C].values                              # Konversi numpy
scaler_C = RobustScaler()                                       # Inisialisasi scaler
X_host = scaler_C.fit_transform(X_host_raw)                      # Fit + transform

print(f"Matriks fitur C : {X_host.shape}")
print(f"Fitur: {features_C}")
# Distribusi listings_per_host
host_counts = df['calculated_host_listings_count'].value_counts().sort_index()
print(f"\nDistribusi listings per host:\n{host_counts.head(10)}")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT C — Silhouette Scan
# ———————————————————————————————————————————————————————————————————————

scores_C, best_k_C, _ = evaluate_k_silhouette(
    X_host, k_range, random_state=RANDOM_STATE
)

plot_silhouette_scan(
    k_range=k_range,
    scores=scores_C,
    best_k=best_k_C,
    title='Kandidat C: Silhouette Score — Host Archetypes'
)


In [ ]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# KANDIDAT C -- Model Fit: KMeans + Agglomerative (Scipy)
# ----------------------------------------------------------------------------------------------------------------------------------------------
# NOTE: sklearn.AgglomerativeClustering pada 48k sampel memakai ~5-10 GB RAM
#       (connectivity matrix n x n).  Solusi: pakai scipy.cluster.hierarchy
#       yang hanya menyimpan (n-1) x 4 linkage matrix -> hemat 100x.

# ---- KMeans (sama seperti sebelumnya, tambah sample_size) ----
kmeans_C = KMeans(n_clusters=best_k_C, random_state=RANDOM_STATE, n_init=10)  # Init KMeans
labels_C = kmeans_C.fit_predict(X_host)               # Fit + prediksi label KMeans
sil_C = silhouette_score(X_host, labels_C, sample_size=10000)  # Silhouette KMeans (sampel 10000)

# ---- Agglomerative via Scipy pada subsample 5000 (hemat memori) ----
from scipy.cluster.hierarchy import linkage as scipy_linkage     # Scipy linkage (O(n) memory)
from scipy.cluster.hierarchy import fcluster                     # Cut tree ke k klaster

n_agg_sample = 5000                                              # Ukuran subsample (aman utk 8-16 GB RAM)
np.random.seed(RANDOM_STATE)                                     # Seed utk reproducibility
idx_agg = np.random.choice(len(X_host), n_agg_sample, replace=False)  # Pilih indeks acak
X_agg_sample = X_host[idx_agg]                                   # Ambil subsample fitur

# Scipy linkage: Ward method, tidak bangun connectivity matrix
Z_C = scipy_linkage(X_agg_sample, method='ward')                  # (n-1) x 4 linkage matrix

# Potong tree pada k klaster (return label 0-indexed)
labels_agg_C_sample = fcluster(Z_C, t=best_k_C, criterion='maxclust') - 1  # Label Agglomerative

# Silhouette Agglomerative pada subsample yang SAMA
sil_agg_C = silhouette_score(X_agg_sample, labels_agg_C_sample)   # Silhouette Agglomerative

# ARI: bandingkan KMeans vs Agglomerative pada SUBSAMPLE yang SAMA
labels_C_sample = labels_C[idx_agg]                               # Ambil KMeans labels utk subsample
ari_C = adjusted_rand_score(labels_C_sample, labels_agg_C_sample)  # Adjusted Rand Index

print(f"   KMeans k={best_k_C}                 -- Silhouette = {sil_C:.4f}")
print(f"   Agglomerative (scipy, n={n_agg_sample}) k={best_k_C} -- Silhouette = {sil_agg_C:.4f}")
print(f"   ARI (KMeans vs Agglom)              = {ari_C:.4f}")
print(f"   Note: Agglomerative dihitung pada subsample {n_agg_sample} -- ARI dihitung pada subsample yg sama.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT C — Dendrogram (visualisasi hierarki)
# ———————————————————————————————————————————————————————————————————————

fig, ax = plt.subplots(figsize=(14, 6))                        # Lebar figure dendrogram
# Gunakan sampel acak agar dendrogram tidak terlalu padat
n_sample = min(2000, len(X_host))                              # Maks 2000 sampel
np.random.seed(RANDOM_STATE)                                   # Reproducible
idx_sample = np.random.choice(len(X_host), n_sample, replace=False)  # Indeks sampel
X_sample = X_host[idx_sample]                                  # Ambil subsample

Z = linkage(X_sample, method='ward')                           # Hitung linkage matrix (Ward)
dendrogram(Z, ax=ax, truncate_mode='level', p=5,              # Dendrogram (truncate untuk readability)
           leaf_font_size=8, color_threshold=Z[-(best_k_C-1), 2])  # Warna sesuai k optimal
ax.set_title(f'Dendrogram Host — Agglomerative (Ward) — {n_sample} sampel')  # Judul
ax.set_xlabel('Indeks Sampel')                                 # Label x
ax.set_ylabel('Jarak Ward')                                    # Label y
plt.tight_layout()
plt.show()
print(f"Dendrogram menunjukkan struktur hierarki host. k={best_k_C} ditandai dengan warna.")
print("Jika cabang-cabang terpisah jelas → arketipe memang diskret & hierarkis.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT C — Peta + Catatan
# ———————————————————————————————————————————————————————————————————————

plot_map_clusters(
    df, labels_C,                                                       # Data + label KMeans
    title=f'Kandidat C — Host Archetypes (k={best_k_C}, KMeans)'        # Judul
)

from IPython.display import display, Markdown
display(Markdown(f"""
> **Catatan interpretasi peta (Kandidat C):**
>
> Dimensi host (jumlah listing, min nights, aktivitas review) **bukan dimensi geografis**,
> sehingga peta ini hanya menunjukkan *di mana* arketipe tertentu muncul.
> **Validasi utama kandidat ini ada pada:**
> - Tabel profil arketipe (bawah)
> - Dendrogram (atas)
> - Silhouette Score = {sil_C:.4f}
>
> Klaster yang tampak "bercampur" di peta **tidak berarti gagal** — artinya
> arketipe host tersebar merata di seluruh NYC (observasi yang menarik secara bisnis).
"""))


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT C — Tabel Profil Arketipe Host
# ———————————————————————————————————————————————————————————————————————

profile_features_C = [
    'calculated_host_listings_count', 'minimum_nights_capped',
    'number_of_reviews', 'last_review_age_days',
    'price', 'availability_365_capped'
]

profile_C = profile_clusters(df, labels_C, profile_features_C)

# Beri nama arketipe berdasarkan karakteristik
profile_C = profile_C.sort_values('calculated_host_listings_count')  # Urut: individu → profesional

display(profile_C.style
    .format("{:.1f}", subset=profile_features_C)
    .background_gradient(cmap='Greens', subset=profile_features_C)
    .set_caption(f'Kandidat C — Profil Host Archetypes (k={best_k_C})')
)

# Interpretasi otomatis
most_hosts = profile_C['calculated_host_listings_count'].idxmax()
fewest_hosts = profile_C['calculated_host_listings_count'].idxmin()
print(f"Silhouette Host : {sil_C:.4f}")
print(f"Arketipe paling 'profesional': klaster {most_hosts} (median {profile_C.loc[most_hosts, 'calculated_host_listings_count']:.0f} listing)")
print(f"Arketipe paling 'individual' : klaster {fewest_hosts} (median {profile_C.loc[fewest_hosts, 'calculated_host_listings_count']:.0f} listing)")


### KANDIDAT D — Multi-Dimensional Market Segmentation ⭐ (Rekomendasi Utama)

**Perspektif Akademis:**

Segmentasi terintegrasi yang menggabungkan **semua dimensi** adalah pendekatan paling
komprehensif untuk memahami pasar Airbnb NYC.  Berbeda dari kandidat A–C yang fokus
pada satu dimensi, kandidat D mengakui bahwa pasar nyata adalah interaksi dari
*lokasi, harga, tipe properti, demand, dan perilaku host* secara simultan.

**Landasan teori:**
- *Porter (1980)* — Competitive strategy: perusahaan/unikorn mengoptimalkan posisi
  dalam *multi-dimensional strategic space*.
- *Smith (1956)* — "Product differentiation and market segmentation as alternative
  marketing strategies": segmentasi multi-variat adalah bentuk paling matang.
- *Chica-Olmo, González-Morales & Zafra-Gómez (2020)* — Studi multi-kriteria pada
  34.000 listing Airbnb menunjukkan bahwa kombinasi fitur spasial + ekonomi + servis
  menghasilkan segmen pasar yang lebih informatif daripada pendekatan univariat.

**Pertanyaan penelitian:**
Segmen pasar apa yang terbentuk jika semua dimensi digunakan? Apakah segmen ini
*masuk akal secara geografis* (validasi koherensi spasial)?

**Fitur (semua dimensi):**
- Geografis: `latitude`, `longitude`, `distance_to_center_km` + `neighbourhood_group` (one-hot, drop_first)
- Properti: `log_price`, `log_min_nights`, `availability_365_capped` + `room_type` (one-hot, drop_first)
- Demand: `log_number_of_reviews`, `last_review_age_days`
- Host: `calculated_host_listings_count`

**Algoritma:**
- **KMeans** — baseline industri.
- **Gaussian Mixture Model (GMM)** — memungkinkan klaster ellipsoidal (tidak memaksa spherical).

**Preprocessing:** `RobustScaler` untuk numerik; `OneHotEncoder` untuk kategorikal.

**Visualisasi:** PCA 2D (hanya untuk plot, bukan untuk fit model).

**Metrik evaluasi:** Silhouette Score untuk KMeans & GMM.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Persiapan Data Multi-Dimensi
# ———————————————————————————————————————————————————————————————————————

from sklearn.preprocessing import OneHotEncoder                  # Encoding kategorikal
from sklearn.compose import ColumnTransformer                     # Pipeline kolom

# --- Fitur numerik ---
num_features_D = [
    'latitude', 'longitude', 'distance_to_center_km',            # Geografis (3)
    'log_price', 'log_min_nights', 'availability_365_capped',    # Properti (3)
    'log_number_of_reviews', 'last_review_age_days',             # Demand (2)
    'calculated_host_listings_count',                             # Host (1)
]

# --- Fitur kategorikal ---
cat_features_D = [
    'neighbourhood_group',                                        # 5 borough
    'room_type',                                                  # 3 tipe (Entire, Private, Shared)
]

# --- Pipeline: RobustScaler untuk numerik, OneHot untuk kategorikal ---
preprocessor_D = ColumnTransformer([
    ('num', RobustScaler(), num_features_D),                     # Numerik → RobustScaler
    ('cat', OneHotEncoder(drop='first', sparse_output=False),     # Kategorikal → OneHot (drop first)
     cat_features_D),
], remainder='drop')                                              # Hapus kolom yg tidak disebut

X_multi = preprocessor_D.fit_transform(df)                        # Fit + transform

# Ambil nama kolom output untuk interpretasi
cat_encoder = preprocessor_D.named_transformers_['cat']           # Ambil OneHotEncoder
cat_col_names = cat_encoder.get_feature_names_out(cat_features_D) # Nama kolom hasil one-hot
all_feature_names = list(num_features_D) + list(cat_col_names)    # Gabungkan semua nama
print(f"Matriks fitur D : {X_multi.shape}")                       # Dimensi
print(f"Total fitur     : {len(all_feature_names)}")               # Jumlah kolom
print(f"Numerik: {len(num_features_D)} | Kategorikal: {len(cat_col_names)}")  # Breakdown


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — PCA (hanya untuk visualisasi 2D, bukan untuk model)
# ———————————————————————————————————————————————————————————————————————

# --- Fit PCA pada data multi-dimensi ---
pca_D = PCA(random_state=RANDOM_STATE)                           # Inisialisasi PCA
pca_D.fit(X_multi)                                               # Fit (tanpa transform dulu)

# --- Plot explained variance ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))           # 2 panel

# Panel 1: Scree plot
ax1.bar(range(1, len(pca_D.explained_variance_ratio_) + 1),     # Bar plot
        pca_D.explained_variance_ratio_, color='steelblue')
ax1.set_xlabel('Komponen Utama')                                 # Label x
ax1.set_ylabel('Varians Dijelaskan (rasio)')                     # Label y
ax1.set_title('Scree Plot — PCA')                                # Judul

# Panel 2: Kumulatif
cumsum = np.cumsum(pca_D.explained_variance_ratio_)               # Hitung kumulatif
ax2.plot(range(1, len(cumsum) + 1), cumsum, marker='o', color='crimson')  # Garis kumulatif
ax2.axhline(y=0.80, color='gray', linestyle='--', alpha=0.5)     # Garis 80%
ax2.axhline(y=0.90, color='gray', linestyle='--', alpha=0.5)     # Garis 90%
ax2.set_xlabel('Jumlah Komponen')                                 # Label x
ax2.set_ylabel('Varians Kumulatif')                               # Label y
ax2.set_title('Varians Kumulatif — PCA')                          # Judul
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cari jumlah komponen untuk ≥85% varians
n85 = int(np.argmax(cumsum >= 0.85)) + 1                          # Komponen untuk 85%
print(f"PC untuk 80% varians : {int(np.argmax(cumsum >= 0.80)) + 1}")
print(f"PC untuk 85% varians : {n85}")
print(f"PC untuk 90% varians : {int(np.argmax(cumsum >= 0.90)) + 1}")
print("PCA hanya digunakan untuk visualisasi 2D — model di-fit pada fitur asli.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Silhouette Scan: KMeans + GMM
# ———————————————————————————————————————————————————————————————————————

scores_D_kmeans = []                                             # Skor untuk KMeans
scores_D_gmm = []                                                # Skor untuk GMM

for k in k_range:                                                # Iterasi k=2..10
    # --- KMeans ---
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_km = km.fit_predict(X_multi)                          # Fit KMeans
    sil_km = silhouette_score(X_multi, labels_km, sample_size=10000)  # Silhouette KMeans
    scores_D_kmeans.append(sil_km)                               # Simpan

    # --- GMM ---
    gmm = GaussianMixture(n_components=k, random_state=RANDOM_STATE, n_init=5)
    labels_gmm = gmm.fit_predict(X_multi)                        # Fit GMM
    sil_gmm = silhouette_score(X_multi, labels_gmm, sample_size=10000)  # Silhouette GMM
    scores_D_gmm.append(sil_gmm)                                 # Simpan

# Tentukan k optimal untuk masing-masing
best_k_D_kmeans = k_range[int(np.argmax(scores_D_kmeans))]
best_k_D_gmm = k_range[int(np.argmax(scores_D_gmm))]

# Plot side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(k_range, scores_D_kmeans, marker='o', color='steelblue', linewidth=2)
ax1.axvline(x=best_k_D_kmeans, color='crimson', linestyle='--')
ax1.scatter(best_k_D_kmeans, max(scores_D_kmeans), color='crimson', s=100, zorder=5)
ax1.set_title(f'KMeans — k optimal = {best_k_D_kmeans}')
ax1.set_xlabel('Jumlah Klaster (k)')
ax1.set_ylabel('Silhouette Score')
ax1.set_xticks(k_range)
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, scores_D_gmm, marker='s', color='darkorange', linewidth=2)
ax2.axvline(x=best_k_D_gmm, color='crimson', linestyle='--')
ax2.scatter(best_k_D_gmm, max(scores_D_gmm), color='crimson', s=100, zorder=5)
ax2.set_title(f'GMM — k optimal = {best_k_D_gmm}')
ax2.set_xlabel('Jumlah Klaster (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(k_range)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"KMeans : k={best_k_D_kmeans}  | Silhouette = {max(scores_D_kmeans):.4f}")
print(f"GMM    : k={best_k_D_gmm}  | Silhouette = {max(scores_D_gmm):.4f}")
print("Pilih algoritma dengan Silhouette tertinggi untuk model final.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Model Final (pilih algoritma terbaik)
# ———————————————————————————————————————————————————————————————————————

# Pilih algoritma terbaik berdasarkan Silhouette
if max(scores_D_kmeans) >= max(scores_D_gmm):
    model_D = KMeans(n_clusters=best_k_D_kmeans, random_state=RANDOM_STATE, n_init=10)
    labels_D = model_D.fit_predict(X_multi)
    algo_D = f'KMeans (k={best_k_D_kmeans})'
    sil_D = max(scores_D_kmeans)
else:
    model_D = GaussianMixture(n_components=best_k_D_gmm, random_state=RANDOM_STATE, n_init=5)
    labels_D = model_D.fit_predict(X_multi)
    algo_D = f'GMM (k={best_k_D_gmm})'
    sil_D = max(scores_D_gmm)

print(f"Model terbaik : {algo_D}")
print(f"Silhouette    : {sil_D:.4f}")
print(f"Jumlah klaster: {len(set(labels_D))}")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Scatter PCA 2D (visualisasi ruang fitur)
# ———————————————————————————————————————————————————————————————————————

# Proyeksikan data multi-dimensi ke 2 PC pertama
X_pca2d = pca_D.transform(X_multi)[:, :2]                       # Ambil 2 komponen pertama

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    X_pca2d[:, 0], X_pca2d[:, 1],                               # PC1 vs PC2
    c=labels_D, cmap='tab10', s=2, alpha=0.5                    # Warna per klaster
)
ax.set_xlabel(f'PC1 ({pca_D.explained_variance_ratio_[0]:.1%} varians)')  # Label PC1
ax.set_ylabel(f'PC2 ({pca_D.explained_variance_ratio_[1]:.1%} varians)')  # Label PC2
ax.set_title(f'Kandidat D — PCA 2D ({algo_D}, Silhouette={sil_D:.3f})')   # Judul
legend = ax.legend(*scatter.legend_elements(), title='Klaster', loc='upper right')
ax.add_artist(legend)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Scatter plot 2D dari PCA. Klaster yang terpisah jelas di PC1/PC2 → validitas tinggi.")
print("Jika klaster tumpang tindih → beberapa dimensi mungkin tidak membedakan dengan baik.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Peta NYC dengan Klaster Multi-Dimensi
# ———————————————————————————————————————————————————————————————————————

plot_map_clusters(
    df, labels_D,                                                    # Data asli + label klaster
    title=f'Kandidat D — Multi-Dimensional ({algo_D}, Silhouette={sil_D:.3f})',
    s=1, alpha=0.5
)
print("Peta NYC dengan warna klaster multi-dimensi.")
print("Jika klaster terlihat 'rapi' secara geografis → validasi koherensi spasial berhasil.")
print("Jika bercampur → artinya dimensi ekonomi/perilaku lebih dominan daripada lokasi.")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Heatmap Profil Klaster (z-score)
# ———————————————————————————————————————————————————————————————————————

# Buat z-score dari median fitur numerik per klaster
profile_features_D = [
    'price', 'distance_to_center_km', 'number_of_reviews',
    'availability_365_capped', 'calculated_host_listings_count',
    'minimum_nights_capped'
]

p = df.copy()                                                      # Copy dataframe
p['cluster'] = labels_D                                            # Tambah kolom label
profile_D_raw = p.groupby('cluster')[profile_features_D].median()  # Median per klaster

# Z-score: (median - mean_all) / std_all  → berapa std di atas/bawah rata-rata
z_profile = (profile_D_raw - profile_D_raw.mean()) / profile_D_raw.std()  # Hitung z-score

fig, ax = plt.subplots(figsize=(14, max(4, len(z_profile) * 0.7)))  # Figure proporsional
sns.heatmap(
    z_profile,                                                      # Matriks z-score
    annot=profile_D_raw.round(1),                                    # Anotasi: median asli
    fmt='', cmap='RdBu_r', center=0,                                 # Colormap diverging
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Z-score'}            # Style
)
ax.set_title(f'Kandidat D — Heatmap Profil Klaster ({algo_D})')     # Judul
ax.set_xlabel('Fitur')                                               # Label x
ax.set_ylabel('Klaster')                                             # Label y
plt.tight_layout()
plt.show()
print("Heatmap: warna merah → di atas rata-rata, biru → di bawah rata-rata.")
print("Cari klaster dengan profil unik (misal: satu klaster merah di price tapi biru di reviews = mahal-sepi).")


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# KANDIDAT D — Tabel Profil Lengkap
# ———————————————————————————————————————————————————————————————————————

feature_names_D = num_features_D + cat_features_D                  # Semua nama fitur

# Ambil median + count + pct
profile_D = profile_clusters(df, labels_D,
    ['price', 'distance_to_center_km', 'number_of_reviews',
     'availability_365_capped', 'calculated_host_listings_count',
     'minimum_nights_capped', 'log_price', 'last_review_age_days'])

# Tambah stats kategorikal per klaster (proporsi room_type & borough)
for room in df['room_type'].unique():
    profile_D[f'room_{room}'] = p.groupby('cluster')['room_type'].apply(
        lambda x: (x == room).mean() * 100).round(1)               # % per klaster

for boro in df['neighbourhood_group'].unique():
    profile_D[f'boro_{boro}'] = p.groupby('cluster')['neighbourhood_group'].apply(
        lambda x: (x == boro).mean() * 100).round(1)               # % per klaster

display(profile_D.style
    .format("{:.1f}")
    .background_gradient(cmap='RdYlGn', axis=0)
    .set_caption(f'Kandidat D — Profil Lengkap ({algo_D}, Silhouette={sil_D:.3f})')
)

print(f"Model final  : {algo_D}")
print(f"Silhouette   : {sil_D:.4f}")
print(f"Jumlah klaster : {len(set(labels_D))}")


---

## RINGKASAN PERBANDINGAN 4 KANDIDAT

Tabel di bawah merangkum seluruh hasil klastering dari keempat perspektif.
Gunakan tabel ini untuk memilih pendekatan mana yang paling sesuai dengan
pertanyaan bisnis / analitis Anda.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# TABEL RINGKASAN — Perbandingan 4 Kandidat
# ———————————————————————————————————————————————————————————————————————

# Kumpulkan semua metrik
summary_data = {
    'Kandidat': ['A: Geographic', 'B: Price Tier', 'C: Host', 'D: Multi-Dim'],
    'Algoritma': [
        f'KMeans (k={best_k_A}) / DBSCAN',
        f'KMeans (k={best_k_B})',
        f'KMeans (k={best_k_C}) / Agglomerative',
        f'{algo_D}',
    ],
    'k Optimal': [best_k_A, best_k_B, best_k_C,
                  best_k_D_kmeans if 'KMeans' in algo_D else best_k_D_gmm],
    'Silhouette': [
        f'{scores_A[best_k_A - k_range[0]]:.4f}',
        f'{scores_B[best_k_B - k_range[0]]:.4f}',
        f'{scores_C[best_k_C - k_range[0]]:.4f}',
        f'{sil_D:.4f}',
    ],
    'Dimensi': [
        'Hanya spasial',
        'Ekonomi + lokasi',
        'Perilaku host',
        'Semua dimensi (9 fitur)',
    ],
    'Fitur Utama': [
        'latitude, longitude',
        'log_price, availability, reviews, jarak',
        'listings_count, min_nights, reviews, recency',
        'numerik + one-hot room_type & neighbourhood_group',
    ],
    'Peta Valid?': [
        'Ya — output utama',
        'Ya — korelasi harga-lokasi',
        'Tambahan — bukan validasi utama',
        'Ya — uji koherensi spasial',
    ],
}

summary_df = pd.DataFrame(summary_data)                          # Buat DataFrame
display(summary_df.style
    .set_caption('Perbandingan 4 Kandidat Clustering')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#4472C4'), ('color', 'white'),
                  ('font-weight', 'bold'), ('text-align', 'center')]
    }])
)


### Rekomendasi Penggunaan

| Jika tujuan Anda... | Gunakan |
|---|---|
| Memahami zonasi alami di NYC (berbeda dari borough) | **Kandidat A** |
| Menentukan strategi harga berbasis data | **Kandidat B** |
| Mengidentifikasi host profesional vs individu | **Kandidat C** |
| Segmentasi pasar yang komprehensif & holistik | **Kandidat D** ⭐ |

**Kandidat D adalah rekomendasi utama** karena:
1. Menangkap semua dimensi pasar (geografis, ekonomi, properti, host) secara simultan.
2. Hasil klaster langsung bisa dipakai untuk: rekomendasi harga, strategi investasi host,
   identifikasi listing outlier, dan analisis kompetitif per borough.
3. Validasi silang: peta + PCA 2D + heatmap + profil → coherence check multi-angle.

**Catatan:** Jika Silhouette Score < 0.25 pada kandidat manapun, struktur klaster dianggap
lemah (data cenderung kontinu tanpa pemisahan natural). Ini adalah temuan yang valid
secara akademis — bukan kegagalan model.


---

## PETA KOMPARATIF — Semua Kandidat dalam Satu Gambar

Keempat peta berdampingan dalam satu figure 2×2 untuk perbandingan visual langsung.


In [ ]:
# ———————————————————————————————————————————————————————————————————————
# PETA KOMPARATIF — 2x2 Subplot
# ———————————————————————————————————————————————————————————————————————

fig, axes = plt.subplots(2, 2, figsize=(18, 16))               # Grid 2x2
axes = axes.flatten()                                           # Flatten ke 1D array

# Data untuk 4 panel
map_data = [
    (labels_kmeans_A, f'A: Geographic (KMeans, k={best_k_A})'),
    (labels_B,        f'B: Price Tier (KMeans, k={best_k_B})'),
    (labels_C,        f'C: Host Archetypes (KMeans, k={best_k_C})'),
    (labels_D,        f'D: Multi-Dimensional ({algo_D})'),
]

for ax, (lbl, title) in zip(axes, map_data):                   # Iterasi 4 panel
    sc = ax.scatter(df['longitude'], df['latitude'],            # Plot titik
                    c=lbl, cmap='tab10', s=0.5, alpha=0.5)
    ax.set_title(title, fontsize=11)                            # Judul panel
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    lat_m = df['latitude'].mean()                               # Rata-rata lat
    ax.set_aspect(1 / np.cos(np.radians(lat_m)))                # Aspect ratio geografis
    # Legend (compact)
    leg = ax.legend(*sc.legend_elements(), title='Klaster',
                    loc='upper right', fontsize=7, title_fontsize=8)
    ax.add_artist(leg)

plt.suptitle('Perbandingan 4 Perspektif Clustering — NYC Airbnb 2019',
             fontsize=14, fontweight='bold', y=1.01)            # Super-title
plt.tight_layout()
plt.show()
print("Keempat peta dalam satu figure. Bandingkan sebaran klaster antar perspektif.")
print("Kandidat A → zona geografis; B → tier harga; C → arketipe host; D → semua dimensi.")


### Insight Komparatif Peta

- **Kandidat A** menunjukkan bahwa KMeans membagi NYC menjadi zona konsentris — sesuai
  dengan ekspektasi urban geography.
- **Kandidat B** memperlihatkan bahwa "premium" tidak selalu di Manhattan; beberapa
  listing di Brooklyn/Queens juga masuk tier atas (menarik untuk analisis lebih lanjut).
- **Kandidat C** konfirmasi bahwa arketipe host tersebar merata — tidak ada borough
  yang "dimonopoli" host profesional.
- **Kandidat D** menggabungkan semua dimensi; visual di peta menunjukkan apakah
  segmentasi multi-dimensi konsisten dengan geografi (coherence test).

---

**Akhir notebook.**
